In [11]:
import numpy as np
import CoRT_builder
import utils
from sklearn.linear_model import Lasso
from scipy.stats import norm


In [12]:
n_target = 50
n_source = 60
p = 100
K = 7
Ka = 2
h = 15
alpha = 0.05
T = 5
s_len = 10
s_vector = [0.5] * s_len

N = n_target + Ka * n_source
NI = n_target + n_source
lamda_k_source = 1.2 * np.sqrt(np.log(p)/ N)
lamda_1_source = 1.2 * np.sqrt(np.log(p)/ NI) 
lamda_not_source = 1.2 * np.sqrt(np.log(p) / n_target) 

CoRT_model = CoRT_builder.CoRT(lamda_not_source)
para_results_storage = []
iterations = 1000

for i in range(0, iterations):
    target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s_len, "AR")

    similar_source_index = CoRT_model.find_similar_source(n_target, K, target_data, source_data, lamda_not_source, lamda_1_source, T=T, verbose=False)
    X_combined, y_combined = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data)
    model = Lasso(alpha=lamda_k_source, fit_intercept=False, tol=1e-10, max_iter=100000)
    model.fit(X_combined, y_combined.ravel())
    beta_hat_target = model.coef_[-p:]

    M_obs = np.array([i for i, b in enumerate(beta_hat_target) if b != 0])
    if len(M_obs) == 0:
        print(f"Iteration {iter}: Lasso selected no features. Skipping.")
        continue

    j = np.random.choice(len(M_obs))
    selected_feature_index = M_obs[j]

    X_target = X_combined
    y_target = y_combined
    X_active, X_inactive = utils.get_active_X(beta_hat_target, X_target)
    etaj, etajTy = utils.construct_test_statistic(y_target, j, X_active)
    n_new = X_target.shape[0]
    Sigma = np.eye(n_new)
    tn_sigma = (np.sqrt(etaj.T @ Sigma @ etaj)).item()

    p_value = 2 * (1 - norm.cdf(abs(etajTy), loc = 0, scale = tn_sigma))

    is_signal = (selected_feature_index < s_len) 
    result_dict = {
        "p_value": p_value,
        "is_signal": is_signal,
        "feature_idx": selected_feature_index
    }
    # print(f"is_signal : {result_dict['is_signal']}, p_values[{i}]: {result_dict['p_value']}")
    para_results_storage.append(result_dict)

is_signal_cases = [r for r in para_results_storage if r['is_signal']]
not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)
print(f"len not_signal_cases : {len(not_signal_cases)}")
print(f"false_positives: {false_positives}")
fpr = false_positives / len(not_signal_cases)
print(f"FPR: {fpr:.4f} (Target: {alpha})")
print("\n")
true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha)
print(f"len signal_cases : {len(is_signal_cases)}")
print(f"true_positives: {true_positives}")
tpr = true_positives / len(is_signal_cases)
print(f"TPR: {tpr:.4f}")


len not_signal_cases : 132
false_positives: 24
FPR: 0.1818 (Target: 0.05)


len signal_cases : 868
true_positives: 639
TPR: 0.7362
